In [13]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

In [14]:
from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------------------------
# Stage 7 output directory
# ------------------------------------------------------------------

stage7_dir = (
        DATA_ROOT
        / "processed"
        / "stage_8_cell_tracking"
)

# ------------------------------------------------------------------
# Load detections
# ------------------------------------------------------------------

detections = pd.read_csv(
    stage7_dir / "detections.csv"
)

# ------------------------------------------------------------------
# Load tracks
# ------------------------------------------------------------------

tracks = pd.read_csv(
    stage7_dir / "tracks.csv"
)

# ------------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------------

with open(stage7_dir / "metadata.json") as f:
    metadata = json.load(f)

print(f"Loaded {len(detections):,} detections")
print(f"Loaded {len(tracks):,} track records")
print(metadata)

Loaded 4,223 detections
Loaded 4,223 track records
{'max_distance': 12.0, 'distance_weight': 0.8, 'volume_weight': 0.2, 'max_volume_ratio': 1.5, 'motion_compensation': True, 'global_motion_estimator': 'median (iteratively refined on matched pairs)', 'gap_closing': True, 'max_gap': 2, 'stitch_max_distance': 15.0, 'stitch_max_volume_ratio': 1.5, 'merge_detection': True, 'merge_max_distance': 15.0, 'merge_volume_tolerance': 0.25}


In [15]:
# ============================================================
# Build track summary
# ============================================================

# Add volume information to each tracked point
track_points = (
    tracks.merge(
        detections[
            ["frame", "cell_id", "volume_voxels"]
        ],
        left_on=["frame", "cell"],
        right_on=["frame", "cell_id"],
        how="left",
    )
)

track_summary = (
    track_points
    .sort_values(["track_id", "frame"])
    .groupby("track_id")
    .agg(
        # Lifetime
        start_frame=("frame", "first"),
        end_frame=("frame", "last"),
        length=("frame", "count"),

        # Detection IDs
        start_cell=("cell", "first"),
        end_cell=("cell", "last"),

        # Start position
        start_z=("z", "first"),
        start_y=("y", "first"),
        start_x=("x", "first"),

        # End position
        end_z=("z", "last"),
        end_y=("y", "last"),
        end_x=("x", "last"),

        # Volume
        start_volume=("volume_voxels", "first"),
        end_volume=("volume_voxels", "last"),
    )
    .reset_index()
)

track_summary.head()

,track_id,start_frame,end_frame,length,start_cell,end_cell,start_z,start_y,start_x,end_z,end_y,end_x,start_volume,end_volume
0,0,0,19,19,0,58,0.231183,6.354839,52.204301,15.322957,32.546044,93.211414,440.0,719.0
1,1,0,0,1,1,1,0.153153,6.815315,71.774775,0.153153,6.815315,71.774775,186.0,186.0
2,2,0,0,1,2,2,0.320833,26.012500,65.191667,0.320833,26.012500,65.191667,222.0,222.0
3,3,0,5,6,3,25,1.074627,62.743555,58.230665,5.773535,74.769094,66.487567,240.0,796.0
4,4,0,18,19,4,27,0.364431,98.571429,34.836735,7.040041,160.604723,47.205339,737.0,636.0


In [16]:
# ============================================================
# Build frame indices
# ============================================================

tracks_starting = {}
tracks_ending = {}

for frame, group in track_summary.groupby("start_frame"):
    tracks_starting[frame] = group.copy()

for frame, group in track_summary.groupby("end_frame"):
    tracks_ending[frame] = group.copy()

print(f"{len(tracks_starting)} start-frame groups")
print(f"{len(tracks_ending)} end-frame groups")

20 start-frame groups
20 end-frame groups


In [17]:
# ============================================================
# Candidate parents
# ============================================================

last_frame = detections["frame"].max()

candidate_parents = track_summary[
    track_summary["end_frame"] < last_frame
    ].copy()

print(f"Candidate parents: {len(candidate_parents)}")

candidate_parents.head()

Candidate parents: 637


,track_id,start_frame,end_frame,length,start_cell,end_cell,start_z,start_y,start_x,end_z,end_y,end_x,start_volume,end_volume
1,1,0,0,1,1,1,0.153153,6.815315,71.774775,0.153153,6.815315,71.774775,186.0,186.0
2,2,0,0,1,2,2,0.320833,26.012500,65.191667,0.320833,26.012500,65.191667,222.0,222.0
3,3,0,5,6,3,25,1.074627,62.743555,58.230665,5.773535,74.769094,66.487567,240.0,796.0
4,4,0,18,19,4,27,0.364431,98.571429,34.836735,7.040041,160.604723,47.205339,737.0,636.0
5,5,0,10,11,5,42,0.710526,127.236842,71.768797,9.596935,157.273817,97.234510,343.0,1049.0


In [18]:
BOUNDARY_MARGIN_Z = 3
BOUNDARY_MARGIN_Y = 10
BOUNDARY_MARGIN_X = 10

In [19]:
def touches_boundary(row):
    return (
            row["end_z"] <= BOUNDARY_MARGIN_Z
            or row["end_z"] >= 63 - BOUNDARY_MARGIN_Z
            or row["end_y"] <= BOUNDARY_MARGIN_Y
            or row["end_y"] >= 255 - BOUNDARY_MARGIN_Y
            or row["end_x"] <= BOUNDARY_MARGIN_X
            or row["end_x"] >= 255 - BOUNDARY_MARGIN_X
    )

In [20]:
candidate_parents = candidate_parents[
    ~candidate_parents.apply(touches_boundary, axis=1)
]

In [21]:
print(f"Candidate parents: {len(candidate_parents)}")

candidate_parents.head()

Candidate parents: 174


,track_id,start_frame,end_frame,length,start_cell,end_cell,start_z,start_y,start_x,end_z,end_y,end_x,start_volume,end_volume
3,3,0,5,6,3,25,1.074627,62.743555,58.230665,5.773535,74.769094,66.487567,240.0,796.0
4,4,0,18,19,4,27,0.364431,98.571429,34.836735,7.040041,160.604723,47.205339,737.0,636.0
5,5,0,10,11,5,42,0.710526,127.236842,71.768797,9.596935,157.273817,97.234510,343.0,1049.0
13,13,0,15,15,13,41,1.993144,93.462292,58.957884,11.322052,134.762009,64.520742,559.0,901.0
17,17,0,0,1,17,17,5.240299,223.048259,51.014428,5.240299,223.048259,51.014428,589.0,589.0


In [22]:
print(f"Total tracks: {len(track_summary)}")

track_summary["length"].describe()

Total tracks: 861


count    861.000000
mean       4.904762
std        5.990515
min        1.000000
25%        1.000000
50%        2.000000
75%        6.000000
max       20.000000
Name: length, dtype: float64